# EnergyAwarePath - Multi-Exit Model Training
**ADAPTIVE INFERENCE EXTENSION**  
**Low-Power Embedded Systems** - Assignment 3  
Authors: Pedrini, Bellini

This notebook:
1. Generates synthetic IMU data
2. Extracts features (same formulas used on-device)
3. Builds the dataset combining sensor + branch metadata
4. Trains a small MLP for energy cost prediction
5. Trains a linear regression baseline
6. Converts to int8 quantized TFLite
7. Exports as C header for Arduino deployment

## 1. Setup & Imports

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import os

np.random.seed(42)
tf.random.set_seed(42)
print(f'TensorFlow version: {tf.__version__}')

## 2. Synthetic IMU Data Generation
We simulate 5 conditions: still, tilt, light_shake, heavy_shake, smooth_move.  
Each window is ~1.2s at 100Hz (119 samples).

In [ ]:
def generate_imu_window(condition, num_samples=119):
    t = np.linspace(0, 1.19, num_samples)
    
    if condition == 'still':
        # Real LSM9DS1 noise: very low on acc, ~3 deg/s on gyro
        ax = np.random.normal(0.0, 0.005, num_samples)
        ay = np.random.normal(0.0, 0.005, num_samples)
        az = np.random.normal(1.0, 0.005, num_samples)
        gx = np.random.normal(0.0, 1.5, num_samples)
        gy = np.random.normal(0.0, 1.5, num_samples)
        gz = np.random.normal(0.0, 1.5, num_samples)
    elif condition == 'tilt':
        angle = np.random.uniform(15, 60)
        rad = np.radians(angle)
        ax = np.random.normal(np.sin(rad), 0.01, num_samples)
        ay = np.random.normal(0.0, 0.01, num_samples)
        az = np.random.normal(np.cos(rad), 0.01, num_samples)
        gx = np.random.normal(0.0, 2.0, num_samples)
        gy = np.random.normal(0.0, 2.0, num_samples)
        gz = np.random.normal(0.0, 2.0, num_samples)
    elif condition == 'light_shake':
        ax = np.random.normal(0.0, 0.3, num_samples) + 0.1*np.sin(2*np.pi*5*t)
        ay = np.random.normal(0.0, 0.3, num_samples) + 0.1*np.sin(2*np.pi*4*t)
        az = np.random.normal(1.0, 0.3, num_samples)
        gx = np.random.normal(0.0, 25, num_samples)
        gy = np.random.normal(0.0, 25, num_samples)
        gz = np.random.normal(0.0, 20, num_samples)
    elif condition == 'heavy_shake':
        ax = np.random.normal(0.0, 1.0, num_samples) + 0.5*np.sin(2*np.pi*8*t)
        ay = np.random.normal(0.0, 1.0, num_samples) + 0.5*np.sin(2*np.pi*7*t)
        az = np.random.normal(1.0, 0.8, num_samples)
        gx = np.random.normal(0.0, 100, num_samples)
        gy = np.random.normal(0.0, 100, num_samples)
        gz = np.random.normal(0.0, 80, num_samples)
    elif condition == 'smooth_move':
        ax = 0.2*np.sin(2*np.pi*1.5*t) + np.random.normal(0, 0.05, num_samples)
        ay = 0.15*np.cos(2*np.pi*1.0*t) + np.random.normal(0, 0.05, num_samples)
        az = np.random.normal(0.95, 0.05, num_samples)
        gx = 15*np.sin(2*np.pi*1.5*t) + np.random.normal(0, 3, num_samples)
        gy = 12*np.cos(2*np.pi*1.0*t) + np.random.normal(0, 3, num_samples)
        gz = np.random.normal(0, 4, num_samples)
    
    return ax, ay, az, gx, gy, gz

print('IMU generator ready')

## 3. Feature Extraction
Same 5 features computed on-device: acc_mean, acc_std, acc_peak, gyro_mean, tilt.

In [ ]:
def extract_features(ax, ay, az, gx, gy, gz):
    acc_mag = np.sqrt(ax**2 + ay**2 + az**2)
    acc_mean = np.mean(acc_mag)
    acc_std = np.std(acc_mag)
    acc_peak = np.max(acc_mag)
    
    gyro_mag = np.sqrt(gx**2 + gy**2 + gz**2)
    gyro_mean = np.mean(gyro_mag)
    
    mean_ax = np.mean(ax)
    mean_az = np.mean(az)
    tilt = np.degrees(np.arctan2(abs(mean_ax), abs(mean_az)))
    
    return acc_mean, acc_std, acc_peak, gyro_mean, tilt

# Generate all IMU windows
conditions = ['still', 'tilt', 'light_shake', 'heavy_shake', 'smooth_move']
windows_per_condition = 20  # 100 total

imu_features_list = []
condition_labels = []

for cond in conditions:
    for _ in range(windows_per_condition):
        ax, ay, az, gx, gy, gz = generate_imu_window(cond)
        feats = extract_features(ax, ay, az, gx, gy, gz)
        imu_features_list.append(feats)
        condition_labels.append(cond)

imu_features = np.array(imu_features_list)
print(f'Generated {len(imu_features)} IMU windows')
print(f'Feature ranges:')
for i, name in enumerate(['acc_mean', 'acc_std', 'acc_peak', 'gyro_mean', 'tilt']):
    print(f'  {name}: [{imu_features[:,i].min():.3f}, {imu_features[:,i].max():.3f}]')

## 4. Dataset Construction
Combine IMU features with branch metadata and energy budget.  
Target: energy cost from weighted formula + noise.

In [ ]:
# Branch configurations (4 checkpoints x 3 branches = 12 configs)
branch_configs = [
    (0.3, 5, 0.8, 0.4),   # CP1-A: short, hard, steep
    (0.5, 3, 0.5, 0.1),   # CP1-B: medium
    (0.7, 1, 0.2, -0.1),  # CP1-C: long, easy
    (0.2, 4, 0.9, 0.6),   # CP2-A
    (0.6, 2, 0.4, 0.0),   # CP2-B
    (0.8, 1, 0.1, -0.2),  # CP2-C
    (0.4, 3, 0.7, 0.3),   # CP3-A
    (0.5, 2, 0.5, -0.1),  # CP3-B
    (0.9, 0, 0.1, -0.3),  # CP3-C
    (0.3, 4, 0.6, 0.5),   # CP4-A
    (0.4, 3, 0.4, 0.2),   # CP4-B
    (0.6, 1, 0.3, 0.0),   # CP4-C
]

budget_values = np.linspace(0.2, 1.0, 9)

# Pre-compute sensor feature ranges (used to normalize within formula)
imu_arr = np.array(imu_features_list)
acc_mean_min, acc_mean_max = imu_arr[:,0].min(), imu_arr[:,0].max()
acc_std_min, acc_std_max   = imu_arr[:,1].min(), imu_arr[:,1].max()
acc_peak_min, acc_peak_max = imu_arr[:,2].min(), imu_arr[:,2].max()
gyro_min, gyro_max         = imu_arr[:,3].min(), imu_arr[:,3].max()
tilt_min, tilt_max         = imu_arr[:,4].min(), imu_arr[:,4].max()

def norm(x, lo, hi):
    if hi - lo < 1e-9: return 0.0
    return np.clip((x - lo) / (hi - lo), 0.0, 1.0)

X_data = []
y_data = []

for imu_feat in imu_features_list:
    acc_mean, acc_std, acc_peak, gyro_mean, tilt = imu_feat
    acc_std_n = norm(acc_std, acc_std_min, acc_std_max)
    gyro_n    = norm(gyro_mean, gyro_min, gyro_max)
    
    for branch in branch_configs:
        length, turns, difficulty, slope = branch
        for budget in budget_values:
            feature_vec = [
                budget, length, turns/5.0, difficulty, slope,
                acc_mean, acc_std, acc_peak, gyro_mean, tilt
            ]
            # Energy cost formula with motion-length interaction.
            # Long paths cost MORE under motion -> short paths win when shaken.
            motion = (acc_std_n + gyro_n) / 2.0
            E = (0.18*length + 0.12*(turns/5.0) + 0.12*difficulty +
                 0.08*abs(slope) + 0.05*acc_std_n + 0.05*gyro_n +
                 0.05*(1.0 - budget) + 0.30*length*motion)
            E += np.random.normal(0, 0.02)
            E = np.clip(E, 0.05, 0.95)
            X_data.append(feature_vec)
            y_data.append(E)

X_data = np.array(X_data, dtype=np.float32)
y_data = np.array(y_data, dtype=np.float32)
print(f'Dataset: {X_data.shape[0]} samples, {X_data.shape[1]} features')
print(f'Target range: [{y_data.min():.3f}, {y_data.max():.3f}]')
print(f'Target mean: {y_data.mean():.3f}, std: {y_data.std():.3f}')

## 5. Normalization & Train/Test Split

In [ ]:
# Compute min/max for normalization
feature_min = X_data.min(axis=0)
feature_max = X_data.max(axis=0)
feature_range = feature_max - feature_min
feature_range[feature_range == 0] = 1.0

X_normalized = (X_data - feature_min) / feature_range

# Print normalization constants (needed for Arduino config.h)
feature_labels = ['budget', 'length', 'turns_norm', 'difficulty', 'slope',
                  'acc_mean', 'acc_std', 'acc_peak', 'gyro_mean', 'tilt']
print('Normalization constants for config.h:')
print('const float FEATURE_MIN[10] = {')
print(f'  {", ".join(f"{v:.4f}f" for v in feature_min)}')
print('};')
print('const float FEATURE_MAX[10] = {')
print(f'  {", ".join(f"{v:.4f}f" for v in feature_max)}')
print('};')

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y_data, test_size=0.2, random_state=42)
print(f'\nTrain: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 6. Baseline: Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_mae = mean_absolute_error(y_test, lr_pred)
print(f'Linear Regression MAE: {lr_mae:.4f}')

## 7. MLP Model Training
Architecture: Dense(16, relu) → Dense(8, relu) → Dense(1, sigmoid)  
Output in [0,1] = predicted energy cost.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(10,)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()
print(f'Total parameters: {model.count_params()}')

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=20, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=200,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Evaluate
mlp_pred = model.predict(X_test).flatten()
mlp_mae = mean_absolute_error(y_test, mlp_pred)
print(f'\nMLP MAE:              {mlp_mae:.4f}')
print(f'Linear Regression MAE: {lr_mae:.4f}')
print(f'MLP improvement:       {((lr_mae - mlp_mae)/lr_mae*100):.1f}%')

## 8. Training Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].scatter(y_test, mlp_pred, alpha=0.3, s=10, label='MLP')
axes[1].scatter(y_test, lr_pred, alpha=0.3, s=10, label='Linear')
axes[1].plot([0,1],[0,1],'r--', label='Perfect')
axes[1].set_xlabel('True'); axes[1].set_ylabel('Predicted')
axes[1].set_title('Predictions vs Truth'); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

## 9. TFLite Conversion (int8 quantized)
Full integer quantization for Arduino Nano 33 BLE Sense.

In [ ]:
def representative_dataset_gen():
    indices = np.random.choice(len(X_train), size=100, replace=False)
    for i in indices:
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
print(f'TFLite model size: {len(tflite_model)} bytes ({len(tflite_model)/1024:.1f} KB)')

# Save .tflite file
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)
print('Saved model.tflite')

## 10. Verify Quantized Model Accuracy

In [ ]:
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_scale = input_details[0]['quantization'][0]
input_zp = input_details[0]['quantization'][1]
output_scale = output_details[0]['quantization'][0]
output_zp = output_details[0]['quantization'][1]

print(f'Input:  scale={input_scale:.6f}, zero_point={input_zp}')
print(f'Output: scale={output_scale:.6f}, zero_point={output_zp}')

# Test accuracy
quant_preds = []
for i in range(len(X_test)):
    input_quant = (X_test[i:i+1] / input_scale + input_zp).astype(np.int8)
    interpreter.set_tensor(input_details[0]['index'], input_quant)
    interpreter.invoke()
    out = interpreter.get_tensor(output_details[0]['index'])
    quant_preds.append((out[0][0] - output_zp) * output_scale)

quant_preds = np.array(quant_preds)
quant_mae = mean_absolute_error(y_test, quant_preds)
print(f'\nQuantized MAE: {quant_mae:.4f}')
print(f'Float MAE:     {mlp_mae:.4f}')
print(f'Degradation:   {((quant_mae-mlp_mae)/mlp_mae*100):.1f}%')

## 11. Export as C Header (model.h)
Download this file and place it in the Arduino sketch folder.

In [ ]:
# Generate C header
header_lines = []
header_lines.append('// Auto-generated TFLite model data')
header_lines.append('// Model: Energy cost predictor (10->16->8->1, int8)')
header_lines.append(f'// Size: {len(tflite_model)} bytes')
header_lines.append(f'// Input scale: {input_scale:.6f}, zero_point: {input_zp}')
header_lines.append(f'// Output scale: {output_scale:.6f}, zero_point: {output_zp}')
header_lines.append(f'// Test MAE (quantized): {quant_mae:.4f}')
header_lines.append('')
header_lines.append('#ifndef MODEL_H')
header_lines.append('#define MODEL_H')
header_lines.append('')
header_lines.append(f'const unsigned int model_data_len = {len(tflite_model)};')
header_lines.append('alignas(8) const unsigned char model_data[] = {')

for i in range(0, len(tflite_model), 12):
    chunk = tflite_model[i:i+12]
    hex_vals = ', '.join(f'0x{b:02x}' for b in chunk)
    header_lines.append(f'  {hex_vals},')

header_lines.append('};')
header_lines.append('')
header_lines.append('#endif // MODEL_H')

header_content = '\n'.join(header_lines)
with open('model.h', 'w') as f:
    f.write(header_content)

print('model.h generated!')
print(f'File size: {len(header_content)} bytes')
print('\nDownload model.h and place it in EnergyAwarePath/EnergyAwarePath/')

## 12. Updated config.h Constants
Copy these values into `config.h` on your Arduino sketch.

In [ ]:
print('='*60)
print('UPDATE config.h WITH THESE VALUES:')
print('='*60)
print()
print('// Feature normalization: min values')
print('const float FEATURE_MIN[NUM_FEATURES] = {')
print(f'  {", ".join(f"{v:.4f}f" for v in feature_min)}')
print('};')
print()
print('// Feature normalization: max values')
print('const float FEATURE_MAX[NUM_FEATURES] = {')
print(f'  {", ".join(f"{v:.4f}f" for v in feature_max)}')
print('};')
print()
print(f'// Quantization (read from model at runtime, but for reference):')
print(f'// INPUT_SCALE  = {input_scale:.6f}')
print(f'// INPUT_ZERO   = {input_zp}')
print(f'// OUTPUT_SCALE = {output_scale:.6f}')
print(f'// OUTPUT_ZERO  = {output_zp}')
print()
print('='*60)

## 13. Summary

In [ ]:
print('='*60)
print('TRAINING SUMMARY')
print('='*60)
print(f'Dataset: {X_data.shape[0]} samples, {X_data.shape[1]} features')
print(f'Model: Dense(16,relu) -> Dense(8,relu) -> Dense(1,sigmoid)')
print(f'Parameters: {model.count_params()}')
print(f'TFLite size: {len(tflite_model)} bytes ({len(tflite_model)/1024:.1f} KB)')
print(f'')
print(f'Performance (MAE):')
print(f'  Linear Regression: {lr_mae:.4f}')
print(f'  MLP (float32):     {mlp_mae:.4f}')
print(f'  MLP (int8 quant):  {quant_mae:.4f}')
print(f'')
print(f'Quantization:')
print(f'  Input:  scale={input_scale:.6f}, zp={input_zp}')
print(f'  Output: scale={output_scale:.6f}, zp={output_zp}')
print('='*60)

## 14. Download Files
Run the cell below, then download:
1. `model.h` → place in `EnergyAwarePath/EnergyAwarePath/`
2. `model.tflite` → keep for reference/metrics

In [ ]:
from google.colab import files
files.download('model.h')
files.download('model.tflite')

## 15. Multi-Exit Model Training
Building a model with intermediate exit for adaptive inference.

In [ ]:
print('='*70)
print('MULTI-EXIT MODEL TRAINING')
print('='*70)
print()
print('Building model with intermediate exit...')

# Build multi-exit model
inputs = tf.keras.Input(shape=(10,), name='input')
x1 = tf.keras.layers.Dense(16, activation='relu', name='dense_1')(inputs)

# EXIT 1: lightweight classifier after first layer
exit1_out = tf.keras.layers.Dense(1, activation='sigmoid', name='exit1')(x1)

# Continue to final exit
x2 = tf.keras.layers.Dense(8, activation='relu', name='dense_2')(x1)
final_out = tf.keras.layers.Dense(1, activation='sigmoid', name='final')(x2)

model_multi = tf.keras.Model(inputs=inputs, outputs=[exit1_out, final_out])

model_multi.compile(
    optimizer='adam',
    loss={'exit1': 'mse', 'final': 'mse'},
    loss_weights={'exit1': 0.4, 'final': 1.0},
    metrics={'exit1': 'mae', 'final': 'mae'}
)

model_multi.summary()
print(f'\nTotal parameters: {model_multi.count_params()}')

In [ ]:
early_stop_multi = tf.keras.callbacks.EarlyStopping(
    monitor='val_final_loss', patience=20, restore_best_weights=True, mode='min')

history_multi = model_multi.fit(
    X_train,
    {'exit1': y_train, 'final': y_train},
    validation_data=(X_test, {'exit1': y_test, 'final': y_test}),
    epochs=200,
    batch_size=16,
    callbacks=[early_stop_multi],
    verbose=1
)

print('\nMulti-exit training complete!')

In [ ]:
# Evaluate both exits
exit1_pred, final_pred = model_multi.predict(X_test)
exit1_pred = exit1_pred.flatten()
final_pred = final_pred.flatten()

mae_exit1 = mean_absolute_error(y_test, exit1_pred)
mae_final = mean_absolute_error(y_test, final_pred)

print('='*70)
print('MULTI-EXIT MODEL EVALUATION')
print('='*70)
print(f'Exit 1 MAE:  {mae_exit1:.4f}')
print(f'Final MAE:   {mae_final:.4f}')
print(f'Degradation: {((mae_exit1 - mae_final) / mae_final * 100):.1f}%')
print('='*70)

## 16. Extract and Convert Exit 1 Model

In [ ]:
# Create a model that uses only exit 1 output
exit1_model = tf.keras.Model(
    inputs=model_multi.input,
    outputs=model_multi.get_layer('exit1').output
)

print('Exit 1 model created')
print(f'Parameters: {exit1_model.count_params()}')
exit1_model.summary()

In [ ]:
def representative_dataset_gen_exit1():
    indices = np.random.choice(len(X_train), size=100, replace=False)
    for i in indices:
        yield [X_train[i:i+1].astype(np.float32)]

converter_exit1 = tf.lite.TFLiteConverter.from_keras_model(exit1_model)
converter_exit1.optimizations = [tf.lite.Optimize.DEFAULT]
converter_exit1.representative_dataset = representative_dataset_gen_exit1
converter_exit1.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_exit1.inference_input_type = tf.int8
converter_exit1.inference_output_type = tf.int8

tflite_exit1 = converter_exit1.convert()
print(f'Exit1 TFLite model size: {len(tflite_exit1)} bytes ({len(tflite_exit1)/1024:.2f} KB)')

# Save .tflite file
with open('model_exit1.tflite', 'wb') as f:
    f.write(tflite_exit1)
print('Saved model_exit1.tflite')

In [ ]:
interpreter_exit1 = tf.lite.Interpreter(model_content=tflite_exit1)
interpreter_exit1.allocate_tensors()

input_details_exit1 = interpreter_exit1.get_input_details()
output_details_exit1 = interpreter_exit1.get_output_details()

input_scale_exit1 = input_details_exit1[0]['quantization'][0]
input_zp_exit1 = input_details_exit1[0]['quantization'][1]
output_scale_exit1 = output_details_exit1[0]['quantization'][0]
output_zp_exit1 = output_details_exit1[0]['quantization'][1]

print(f'Exit1 Input:  scale={input_scale_exit1:.6f}, zero_point={input_zp_exit1}')
print(f'Exit1 Output: scale={output_scale_exit1:.6f}, zero_point={output_zp_exit1}')

# Test accuracy
quant_preds_exit1 = []
for i in range(len(X_test)):
    input_quant = (X_test[i:i+1] / input_scale_exit1 + input_zp_exit1).astype(np.int8)
    interpreter_exit1.set_tensor(input_details_exit1[0]['index'], input_quant)
    interpreter_exit1.invoke()
    out = interpreter_exit1.get_tensor(output_details_exit1[0]['index'])
    quant_preds_exit1.append((out[0][0] - output_zp_exit1) * output_scale_exit1)

quant_preds_exit1 = np.array(quant_preds_exit1)
quant_mae_exit1 = mean_absolute_error(y_test, quant_preds_exit1)
print(f'\nExit1 Quantized MAE: {quant_mae_exit1:.4f}')
print(f'Exit1 Float MAE:     {mae_exit1:.4f}')
print(f'Degradation:         {((quant_mae_exit1-mae_exit1)/mae_exit1*100):.1f}%')

## 17. Convert Full Model from Multi-Exit Training

In [ ]:
# Create a model that uses only the final output
full_model = tf.keras.Model(
    inputs=model_multi.input,
    outputs=model_multi.get_layer('final').output
)

print('Full model (from multi-exit training) created')
print(f'Parameters: {full_model.count_params()}')
full_model.summary()

In [ ]:
converter_full = tf.lite.TFLiteConverter.from_keras_model(full_model)
converter_full.optimizations = [tf.lite.Optimize.DEFAULT]
converter_full.representative_dataset = representative_dataset_gen
converter_full.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_full.inference_input_type = tf.int8
converter_full.inference_output_type = tf.int8

tflite_full = converter_full.convert()
print(f'Full TFLite model size: {len(tflite_full)} bytes ({len(tflite_full)/1024:.2f} KB)')

# Save .tflite file
with open('model_full.tflite', 'wb') as f:
    f.write(tflite_full)
print('Saved model_full.tflite')

## 18. Export C Headers

In [ ]:
def tflite_to_c_header(tflite_bytes, var_name, model_type, mae_value):
    """Convert TFLite model to C header format"""
    header_lines = []
    header_lines.append(f'// Auto-generated TFLite model data - {model_type}')
    header_lines.append('// Model: Energy cost predictor (int8)')
    header_lines.append(f'// Size: {len(tflite_bytes)} bytes')
    header_lines.append(f'// Test MAE (quantized): {mae_value:.4f}')
    header_lines.append('')
    header_lines.append(f'#ifndef {var_name.upper()}_H')
    header_lines.append(f'#define {var_name.upper()}_H')
    header_lines.append('')
    header_lines.append(f'const unsigned int {var_name}_len = {len(tflite_bytes)};')
    header_lines.append(f'alignas(8) const unsigned char {var_name}[] = {{')
    
    for i in range(0, len(tflite_bytes), 12):
        chunk = tflite_bytes[i:i+12]
        hex_vals = ', '.join(f'0x{b:02x}' for b in chunk)
        header_lines.append(f'  {hex_vals},')
    
    header_lines.append('};')
    header_lines.append('')
    header_lines.append(f'#endif // {var_name.upper()}_H')
    
    return '\n'.join(header_lines)

print('C header conversion function ready')

In [ ]:
header_exit1 = tflite_to_c_header(
    tflite_exit1, 
    'model_exit1_data', 
    'Exit 1 (input->Dense(16)->Dense(1))',
    quant_mae_exit1
)

with open('model_exit1.h', 'w') as f:
    f.write(header_exit1)

print('model_exit1.h generated!')
print(f'File size: {len(header_exit1)} bytes')

In [ ]:
# Verify full model accuracy first
interpreter_full = tf.lite.Interpreter(model_content=tflite_full)
interpreter_full.allocate_tensors()

input_details_full = interpreter_full.get_input_details()
output_details_full = interpreter_full.get_output_details()

input_scale_full = input_details_full[0]['quantization'][0]
input_zp_full = input_details_full[0]['quantization'][1]
output_scale_full = output_details_full[0]['quantization'][0]
output_zp_full = output_details_full[0]['quantization'][1]

quant_preds_full = []
for i in range(len(X_test)):
    input_quant = (X_test[i:i+1] / input_scale_full + input_zp_full).astype(np.int8)
    interpreter_full.set_tensor(input_details_full[0]['index'], input_quant)
    interpreter_full.invoke()
    out = interpreter_full.get_tensor(output_details_full[0]['index'])
    quant_preds_full.append((out[0][0] - output_zp_full) * output_scale_full)

quant_preds_full = np.array(quant_preds_full)
quant_mae_full = mean_absolute_error(y_test, quant_preds_full)

print(f'Full model quantized MAE: {quant_mae_full:.4f}')

# Generate header
header_full = tflite_to_c_header(
    tflite_full, 
    'model_data', 
    'Full Model (input->Dense(16)->Dense(8)->Dense(1))',
    quant_mae_full
)

with open('model.h', 'w') as f:
    f.write(header_full)

print('model.h generated!')
print(f'File size: {len(header_full)} bytes')

## 19. Final Summary and Download

In [ ]:
print('='*70)
print('MULTI-EXIT MODEL TRAINING SUMMARY')
print('='*70)
print()
print('Model Sizes:')
print(f'  Full model:  {len(tflite_full)} bytes ({len(tflite_full)/1024:.2f} KB)')
print(f'  Exit 1:      {len(tflite_exit1)} bytes ({len(tflite_exit1)/1024:.2f} KB)')
print(f'  Reduction:   {(1 - len(tflite_exit1)/len(tflite_full))*100:.1f}%')
print()
print('Performance (MAE on test set):')
print(f'  Full model (quantized):  {quant_mae_full:.4f}')
print(f'  Exit 1 (quantized):      {quant_mae_exit1:.4f}')
print(f'  Accuracy degradation:    {((quant_mae_exit1 - quant_mae_full) / quant_mae_full * 100):.1f}%')
print()
print('Adaptive Inference Thresholds (from config.h):')
print(f'  Budget >= 0.6: Use full model')
print(f'  Budget 0.3-0.6: Use exit 1')
print(f'  Budget < 0.3: Use oracle formula (no TFLite)')
print()
print('='*70)

In [ ]:
from google.colab import files

print('Downloading files...')
print()
print('1. model.h -> place in EnergyAwarePath/EnergyAwarePath/')
files.download('model.h')

print('2. model_exit1.h -> place in EnergyAwarePath/EnergyAwarePath/')
files.download('model_exit1.h')

print('3. model.tflite -> keep for reference')
files.download('model.tflite')

print('4. model_exit1.tflite -> keep for reference')
files.download('model_exit1.tflite')

print()
print('='*70)
print('NEXT STEPS:')
print('='*70)
print('1. Download all 4 files above')
print('2. Replace model.h and model_exit1.h in your Arduino sketch folder')
print('3. In inference.cpp, uncomment: #include "model_exit1.h"')
print('4. Compile and upload to Arduino Nano 33 BLE Sense')
print('5. Use serial command "6" to activate adaptive policy')
print('6. Observe exit selection based on budget level')
print('='*70)